# Phase 2 – RAG Evaluation Notebook

This notebook evaluates the RAG pipeline using:
- RAGAS metrics (faithfulness, answer relevancy, context recall, context precision)
- LLM-as-a-Judge (Gemini scoring answers 1-5)

**Run Phase 1 first** before this notebook.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)

## Load Evaluation Dataset

In [ ]:
from src.evaluation.ragas_eval import load_eval_dataset

eval_df = load_eval_dataset()
eval_df.head()

In [ ]:
# Distribution of question types
print('Content type distribution:')
print(eval_df['Context_Content_Type'].value_counts())

## Load Pipeline and Run on Eval Set

In [ ]:
from src.rag_pipeline import RAGPipeline
from src.ingestion.chunk_text import load_chunks
from src.evaluation.ragas_eval import run_pipeline_on_eval_set

chunks   = load_chunks()
pipeline = RAGPipeline(retriever_mode='faiss').load(chunks=chunks)

# Run on first 10 questions for a quick test (remove [:10] for full eval)
results = run_pipeline_on_eval_set(pipeline, eval_df.head(10), top_k=5)
print(f'\nRan pipeline on {len(results)} questions')

## RAGAS Evaluation

In [ ]:
from src.evaluation.ragas_eval import run_ragas_evaluation

ragas_scores = run_ragas_evaluation(results)
print('\nScores:')
for k, v in ragas_scores.items():
    print(f'  {k:<30} {v:.4f}')

In [ ]:
# Visualise RAGAS scores as a bar chart
metrics = list(ragas_scores.keys())
scores  = list(ragas_scores.values())
colors  = ['#4CAF50' if s >= 0.7 else '#FF9800' if s >= 0.5 else '#F44336' for s in scores]

plt.bar(metrics, scores, color=colors, edgecolor='white')
plt.ylim(0, 1.0)
plt.axhline(0.7, color='green', linestyle='--', alpha=0.5, label='Good threshold (0.7)')
plt.title('RAGAS Evaluation Scores – Phase 1 (FAISS)')
plt.ylabel('Score (0–1)')
plt.xticks(rotation=20, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

## LLM-as-a-Judge Evaluation

In [ ]:
from src.evaluation.ragas_eval import run_llm_judge_evaluation

judge_df = run_llm_judge_evaluation(results)
judge_df

In [ ]:
# Verdict pie chart
verdict_counts = judge_df['verdict'].value_counts()
colors_pie = {'correct': '#4CAF50', 'partial': '#FF9800', 'incorrect': '#F44336', 'error': '#9E9E9E'}
pie_colors = [colors_pie.get(v, '#9E9E9E') for v in verdict_counts.index]

plt.pie(
    verdict_counts.values,
    labels=verdict_counts.index,
    colors=pie_colors,
    autopct='%1.0f%%',
    startangle=90,
)
plt.title(f'LLM Judge Verdicts | Avg Score: {judge_df["score"].mean():.2f}/5.0')
plt.tight_layout()
plt.show()

## Compare Phase 1 vs Phase 3 (if both have been run)

In [ ]:
import json
from pathlib import Path

# Load saved eval results if they exist
results_dir = Path('../data/processed')

comparisons = {}
for fname in results_dir.glob('eval_ragas_*.json'):
    phase = fname.stem.replace('eval_ragas_', '')
    with open(fname) as f:
        comparisons[phase] = json.load(f)

if comparisons:
    comp_df = pd.DataFrame(comparisons).T
    comp_df.plot(kind='bar', figsize=(12, 5), ylim=(0, 1))
    plt.title('RAGAS Score Comparison Across Phases')
    plt.ylabel('Score')
    plt.xticks(rotation=0)
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()
else:
    print('No saved evaluation results found yet. Run more phases first.')